# Heart Disease UCI Cleveland — eğitim amaçlı sınıflandırma

Yalnızca doğrulanmış gerçek Cleveland verisi kullanılır. Özgün `num` ve türetilen `disease` hedefi model özelliklerine alınmaz. Model seçimi nested CV'de F2 ile yapılır; test eşiği 0.5'tir. Bu çalışma tanı aracı değildir.

Hücreleri sırayla çalıştırın. Colab'da depo zaten varsa kullanılır; mevcut klasör silinmez veya üzerine yazılmaz. Depo yoksa yalnızca yeni hedef klasöre yüklenen ZIP açılır.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

klasor = Path("/content/kalp-github")

if not klasor.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/AlexXenpai/kalp-hastaligi-projesi.git",
        str(klasor)
    ], check=True)

os.chdir(klasor / "heart-disease-uci")
sys.path.insert(0, str(Path.cwd()))

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-q", "-r", "requirements.txt"
], check=True)

print("Hazır!")

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

candidates = [Path.cwd(), Path('/content/heart-disease-uci')]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'train.py').is_file()), None)
if PROJECT_DIR is None:
    target = Path('/content/heart-disease-uci')
    if target.exists():
        raise FileNotFoundError(f'{target} mevcut fakat train.py yok. Klasör korunuyor; doğru proje klasörünü açın veya içeriği elle tamamlayın.')
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('train.py bulunamadı. Notebook’u proje kökünde açın veya Colab’da proje ZIP’ini yükleyin.') from exc
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise ValueError('Proje dosyalarının bulunduğu bir .zip arşivi yükleyin.')
    target.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        root = target.resolve()
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if root not in destination.parents and destination != root:
                raise ValueError('ZIP içinde hedef klasör dışına çıkan bir yol var.')
        archive.extractall(target)
    nested = next((p for p in target.iterdir() if p.is_dir() and (p / 'train.py').is_file()), None)
    PROJECT_DIR = (nested or target).resolve()
    if not (PROJECT_DIR / 'train.py').is_file():
        raise FileNotFoundError('ZIP içinde train.py içeren proje kökü bulunamadı.')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')])
print('Proje:', PROJECT_DIR)

## Gerçek veri denetimi

CSV içeriği hash ile doğrulanır. Eksikler, kategoriler ve birebir tekrarlar raporlanır; imputasyon ve diğer öğrenilen dönüşümler CV içindeki Pipeline'larda fit edilir.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
import train as proje

raw = proje.load_data()
df, audit = proje.clean_data(raw)
print('Kaynak: UCI Heart Disease — processed Cleveland')
print('Boyut:', raw.shape, '| hastalık yok/var:', df.disease.value_counts().sort_index().to_dict())
display(pd.Series(audit, name='Denetim').to_frame())
display(pd.DataFrame(proje.DICTIONARY, columns=['Sütun', 'Anlam', 'Birim / kodlar', 'Tür']))

## Nested CV ve son değerlendirme

5 dış kat tarafsız eğitim performans tahmini için, her dış eğitim fold'unda 3 iç kat GridSearchCV ayarı için kullanılır. F2 recall'ı precision'dan iki kat ağırlıklandırır; precision hâlâ ölçüte dahil olduğundan herkesi pozitif tahmin etmek kendi başına iyi bir seçim değildir. Test verisi seçimde kullanılmaz. Eski holdout önceki çalışmada görüldüğünden yeni test ölçümü bağımsız dış doğrulama değildir.

Aşağıdaki hücre tüm akışı çalıştırır ve gerçek metrikleri `reports/`, eğitilen Pipeline'ı `artifacts/` altına kaydeder. İlk değerlendirme ve yeni sonuçlar raporda ayrı etiketlenir.

In [ ]:
result = proje.run()
display(pd.read_csv(PROJECT_DIR / 'reports' / 'cv_results.csv'))
display(pd.read_csv(PROJECT_DIR / 'reports' / 'baseline_vs_improved.csv'))
display(pd.read_csv(PROJECT_DIR / 'reports' / 'misclassified_examples.csv'))
display(Markdown('Test etiketi: ' + result['test_evaluation_label']))
display(pd.Series(result['test_metrics'], name='Mevcut holdout üzerinde yeniden değerlendirme').to_frame())

## Kaydedilen grafikler

İlk altı grafik eğitim verisi açıklamalarıdır; karşılaştırma grafiğinde outer-fold standart sapmaları gösterilir. Son üç grafik mevcut holdout'un confusion matrix, PR ve ROC eğrileridir. Korelasyon nedensellik, model olasılığı doğrulanmış tıbbi risk değildir.

In [ ]:
from IPython.display import display
for image in sorted((PROJECT_DIR / 'reports' / 'figures').glob('*.png')):
    print(image.name)
    display(Image(filename=str(image)))

print('Raporlar:', PROJECT_DIR / 'reports')
print('Model:', PROJECT_DIR / 'artifacts' / 'heart_pipeline.joblib')